In [1]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 550.6 kB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 24.7 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008644 sha256=2db7a9764bd35d55bbd35f79b8c182c7aabd8daf186db71b36f4aeda828910ce
  Stored in directory: /home/jovyan/.cache/pip/wheels/16/33/a9/f8bff354a182417214933df74dace2a34b02c3e5643e8fac74
Successfully built pyspark


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col, current_timestamp

In [3]:


spark = SparkSession.builder \
    .appName("bronze-layer") \
    .config("spark.jars.packages",
            "org.postgresql:postgresql:42.6.0,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.type", "hadoop") \
    .config("spark.sql.catalog.lakehouse.warehouse", "/tmp/warehouse") \
    .getOrCreate()

print("✅ Spark com Iceberg iniciado!")

Error: LinkageError occurred while loading main class org.apache.spark.launcher.Main
	java.lang.UnsupportedClassVersionError: org/apache/spark/launcher/Main has been compiled by a more recent version of the Java Runtime (class file version 61.0), this version of the Java Runtime only recognizes class file versions up to 55.0
/opt/conda/lib/python3.11/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.bronze")

In [ ]:
df_raw = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/pipeline_transacoes") \
    .option("dbtable", "staging.transacoes_raw") \
    .option("user", "postgres") \
    .option("password", "1234") \
    .load()

df_raw.show(5)

In [ ]:
df_bronze = df_raw.withColumn(
    "data_ingestao",
    current_timestamp()
).withColumn(
    "data_particao",
    to_date(col("data_ingestao"))
)

df_bronze.show(5)

In [ ]:
df_bronze.writeTo("lakehouse.bronze.transacoes") \
    .partitionedBy("data_particao") \
    .createOrReplace()

print("✅ Dados gravados na Bronze")

In [ ]:
df_bronze.writeTo("lakehouse.bronze.transacoes") \
    .partitionedBy("data_particao") \
    .createOrReplace()

print("✅ Dados gravados na Bronze")

In [ ]:

spark.sql("SHOW PARTITIONS lakehouse.bronze.transacoes").show()

In [ ]:
!ls /tmp/warehouse/bronze/transacoes

In [ ]:
!ls /tmp/warehouse/bronze/transacoes